# FineTuneForge — End-to-End Demo

This notebook walks through the agent pipeline: **plan → preprocess → configure → train → evaluate → publish**.

Set `ANTHROPIC_API_KEY` (and `HF_TOKEN` for publishing) in your environment first.

## 1. Inspect the environment

In [ ]:
from finetune_forge.utils.gpu import get_gpu_count, get_available_vram_gb, get_feasible_method

print('GPUs:', get_gpu_count())
print('Free VRAM (GB):', get_available_vram_gb())
print('Method for 8B @ 16GB:', get_feasible_method(8.0, 16.0))

## 2. Create a tiny demo dataset

In [ ]:
import json, pathlib

rows = [
    {'instruction': 'How do I reset my password?', 'output': "Click 'Forgot password' on the login page."},
    {'instruction': 'How do I cancel my subscription?', 'output': 'Go to Billing > Cancel plan.'},
    {'instruction': 'Where are my invoices?', 'output': 'They live under Billing > Invoices.'},
]
path = pathlib.Path('demo_support_qa.json')
path.write_text(json.dumps(rows))
print('wrote', path)

## 3. Run the data processor in isolation

In [ ]:
from finetune_forge.agents.data_processor import run_data_processor

state = {'dataset_path': str(path), 'current_step': 'init', 'error': None}
state = run_data_processor(state)
state['dataset_info']

## 4. Run the full pipeline

This calls Claude for planning/config/judging. Training runs only if a local
LlamaFactory checkout is available via `LLAMAFACTORY_DIR`; otherwise the executor
records an informative error and the graph stops there.

In [ ]:
from finetune_forge.graph.pipeline import run_pipeline

final = run_pipeline(
    task_description='Answer customer support questions for a SaaS product',
    dataset_path=str(path),
    output_hub_repo='',  # leave empty to skip publishing
)
final['model_config'], final.get('error')